[ Input RGB Image Stream ]
                                     │
                                     ▼
                   ┌───────────────────────────────────┐
                   │  DensePose & Mesh 3D Estimator    │
                   │  - Feature Extraction (CNN)       │
                   │  - Differentiable Spatial Softmax │
                   │  - 2D-to-3D Depth Lifting         │
                   └─────────────────┬─────────────────┘
                                     │ 3D Vertex Coordinates & UV Surface Maps
                                     ▼
                   ┌───────────────────────────────────┐
                   │   Spatial-Temporal GNN Encoder    │
                   │   - Graph Convolutions over Topology│
                   │   - 1D Temporal Convolutions      │
                   └─────────────────┬─────────────────┘
                                     │ Condition Vector
                                     ▼
                   ┌───────────────────────────────────┐
                   │  Conditional Latent Diffusion     │
                   │  - Denoising Trajectory Predictor │
                   │  - Probabilistic Motion Sampler   │
                   └─────────────────┬─────────────────┘
                                     │ Forecasted Future Human Poses
                                     ▼
                   ┌───────────────────────────────────┐
                   │  CBF-Shielded Actor-Critic RL     │
                   │  - Policy & Value Prediction      │
                   │  - Control Barrier Function (CBF) │
                   │  - Quadratic Program Shielding    │
                   └─────────────────┬─────────────────┘
                                     │ Safe Joint Velocities
                                     ▼
                          [ Robot Hardware Execution

In [2]:
"""
Module 1: Monocular Three-Dimensional Human Mesh and Dense Surface Estimator
----------------------------------------------------------------------------
Implements a differentiable dual-branch pipeline:
1. Spatial Soft-Argmax for keypoint localization without discrete rounding errors.
2. Continuous UV (Horizontal-Vertical Coordinate Space) surface coordinate map generation 
   (Dense Surface Context Mapping paradigm).
3. Monocular Two-Dimensional to Three-Dimensional Depth Lifting via Multi-Layer Perceptrons.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class DifferentiableSpatialSoftmax(nn.Module):
    """
    Computes sub-pixel Two-Dimensional spatial coordinates from heatmap probability distributions.
    
    Standard argmax operations are non-differentiable. Spatial Softmax calculates 
    the expected value of feature coordinates across spatial dimensions:
    
        Expected Value of x = sum_i sum_j (x_j * Probability(y_i, x_j))
        Expected Value of y = sum_i sum_j (y_i * Probability(y_i, x_j))
        
    where Probability is the spatial Softmax distribution over feature maps.
    """
    def __init__(self, height: int, width: int):
        super(DifferentiableSpatialSoftmax, self).__init__()
        self.height = height
        self.width = width

        # Pre-compute coordinate grids normalized to range [0.0, 1.0]
        grid_x = torch.linspace(0.0, 1.0, width)
        grid_y = torch.linspace(0.0, 1.0, height)
        
        # Grid shapes: [Height, Width]
        grid_y, grid_x = torch.meshgrid(grid_y, grid_x, indexing='ij')
        
        # Reshape to [1, 1, Height, Width] for operational broadcasting
        self.register_buffer('grid_x', grid_x.unsqueeze(0).unsqueeze(0))
        self.register_buffer('grid_y', grid_y.unsqueeze(0).unsqueeze(0))

    def forward(self, heatmaps: torch.Tensor) -> torch.Tensor:
        """
        Args:
            heatmaps: Tensor of shape [Batch_Size, Number_of_Vertices, Height, Width]
        Returns:
            Coordinates Two-Dimensional: Tensor of shape [Batch_Size, Number_of_Vertices, 2] representing (x, y)
        """
        batch_size, num_vertices, h, w = heatmaps.shape

        # Flatten spatial dimensions to compute Softmax probability maps
        # Shape: [Batch_Size, Number_of_Vertices, Height * Width]
        flat_heatmaps = heatmaps.view(batch_size, num_vertices, -1)
        probs = F.softmax(flat_heatmaps, dim=-1)
        probs = probs.view(batch_size, num_vertices, h, w)

        # Compute spatial expectation
        expected_x = torch.sum(probs * self.grid_x, dim=[-2, -1])  # Shape: [Batch_Size, Number_of_Vertices]
        expected_y = torch.sum(probs * self.grid_y, dim=[-2, -1])  # Shape: [Batch_Size, Number_of_Vertices]

        # Concatenate along coordinate axis -> [Batch_Size, Number_of_Vertices, 2]
        return torch.stack([expected_x, expected_y], dim=-1)


class DenseMeshExtractor(nn.Module):
    """
    Backbone Convolutional Neural Network predicting vertex heatmaps and surface UV (Horizontal-Vertical) coordinates.
    """
    def __init__(self, in_channels: int = 3, num_vertices: int = 689, num_parts: int = 24):
        super(DenseMeshExtractor, self).__init__()
        
        # Feature Extraction Backbone (Convolutional Neural Network Pipeline)
        self.backbone = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        # Heatmap branch: maps internal features to vertex probability maps
        self.heatmap_head = nn.Conv2d(256, num_vertices, kernel_size=1)
        
        # Spatial Soft-Argmax layer operating at spatial resolution (56 x 56)
        self.soft_argmax = DifferentiableSpatialSoftmax(height=56, width=56)

        # DensePose (Dense Surface Context Mapping) Branch: Predicts Part Segmentation + (U, V) Continuous Surface Maps
        # Output channels = 24 Body Parts + 2 (U, V channels) = 26 channels
        self.uv_head = nn.Conv2d(256, num_parts + 2, kernel_size=1)

    def forward(self, x: torch.Tensor):
        """
        Args:
            x: Raw input Red-Green-Blue image tensor of shape [Batch_Size, 3, 224, 224]
        Returns:
            coords_2d: Sub-pixel Two-Dimensional vertex coordinates [Batch_Size, Number_of_Vertices, 2]
            uv_surface_map: Dense surface tensor [Batch_Size, Number_of_Parts + 2, 56, 56]
        """
        features = self.backbone(x)                  # [Batch_Size, 256, 56, 56]
        heatmaps = self.heatmap_head(features)        # [Batch_Size, Number_of_Vertices, 56, 56]
        coords_2d = self.soft_argmax(heatmaps)        # [Batch_Size, Number_of_Vertices, 2]
        uv_surface_map = self.uv_head(features)       # [Batch_Size, 26, 56, 56]

        return coords_2d, uv_surface_map


class MonocularDepthLifter(nn.Module):
    """
    Lifts Two-Dimensional pixel coordinates (x, y) into Three-Dimensional camera coordinates (x, y, z).
    """
    def __init__(self, num_vertices: int = 689):
        super(MonocularDepthLifter, self).__init__()
        self.num_vertices = num_vertices

        # Multi-Layer Perceptron architecture for relative depth (z-coordinate) estimation
        self.lifter = nn.Sequential(
            nn.Linear(num_vertices * 2, 1024),
            nn.LayerNorm(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.ReLU(inplace=True),
            
            # Predicts per-vertex relative depth offset (z-coordinate)
            nn.Linear(512, num_vertices)
        )

    def forward(self, coords_2d: torch.Tensor) -> torch.Tensor:
        """
        Args:
            coords_2d: Tensor of shape [Batch_Size, Number_of_Vertices, 2]
        Returns:
            mesh_3d: Full Three-Dimensional Cartesian coordinates tensor [Batch_Size, Number_of_Vertices, 3]
        """
        batch_size = coords_2d.size(0)
        flat_2d = coords_2d.view(batch_size, -1)     # [Batch_Size, Number_of_Vertices * 2]
        
        # Estimate relative depth coordinate z
        pred_z = self.lifter(flat_2d).unsqueeze(-1)  # [Batch_Size, Number_of_Vertices, 1]

        # Concatenate Two-Dimensional plane coordinates with z dimension -> [Batch_Size, Number_of_Vertices, 3]
        mesh_3d = torch.cat([coords_2d, pred_z], dim=-1)
        return mesh_3d


# Verification Block
if __name__ == "__main__":
    print("[Module 1 Verification]")
    batch_img = torch.randn(4, 3, 224, 224) # Batch of 4 Red-Green-Blue image crops
    
    extractor = DenseMeshExtractor(num_vertices=689)
    lifter = MonocularDepthLifter(num_vertices=689)

    c2d, uv_map = extractor(batch_img)
    mesh3d = lifter(c2d)

    print(f"  Input Image Shape : {batch_img.shape}")
    print(f"  Extracted Two-Dimensional Pose : {c2d.shape}")
    print(f"  DensePose UV Surface Maps : {uv_map.shape}")
    print(f"  Final Three-Dimensional Mesh Output: {mesh3d.shape}\n")

[Module 1 Verification]
  Input Image Shape : torch.Size([4, 3, 224, 224])
  Extracted Two-Dimensional Pose : torch.Size([4, 689, 2])
  DensePose UV Surface Maps : torch.Size([4, 26, 56, 56])
  Final Three-Dimensional Mesh Output: torch.Size([4, 689, 3])



In [3]:
"""
Module 2: Spatial-Temporal Graph Neural Network & Conditional Latent Motion Diffusion Predictor
----------------------------------------------------------------------------------------------
Includes:
1. Spatial-Temporal Graph Convolutional Network Block.
2. Latent Diffusion Denoising Network predicting future human pose sequences.
"""

import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv


class SpatialTemporalGraphConv(nn.Module):
    """
    Applies spatial graph convolutions followed by One-Dimensional temporal convolutions over temporal sequences.
    """
    def __init__(self, in_channels: int, out_channels: int, temporal_kernel_size: int = 3):
        super(SpatialTemporalGraphConv, self).__init__()
        
        # Spatial Graph Convolutional Network operating over joint topologies
        self.spatial_gcn = GCNConv(in_channels, out_channels)
        
        # Temporal Convolution operating along time-series dimension
        padding = (temporal_kernel_size - 1) // 2
        self.temporal_conv = nn.Sequential(
            nn.Conv1d(out_channels, out_channels, kernel_size=temporal_kernel_size, padding=padding),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Node feature tensor [Batch_Size, Time_Steps, Number_of_Nodes, Feature_Dimension]
            edge_index: Graph connectivity matrix [2, Number_of_Edges]
        Returns:
            Output Tensor: Processed features [Batch_Size, Time_Steps, Number_of_Nodes, Output_Channels]
        """
        batch_size, time_steps, num_nodes, feature_dim = x.shape

        # 1. Reshape tensor to apply spatial Graph Convolutional Network over nodes per frame
        # Shape: [Batch_Size * Time_Steps, Number_of_Nodes, Feature_Dimension]
        x_flat = x.view(batch_size * time_steps, num_nodes, feature_dim)
        
        # Collapse batch and temporal dimension for batch execution
        # PyTorch Geometric Graph Convolutional Network expects flattened input [Total_Nodes, Feature_Dimension]
        x_gcn_in = x_flat.view(-1, feature_dim)
        
        # Adjust graph indices for batched execution
        # Replicate edge_index across all frames in batch
        total_graphs = batch_size * time_steps
        batch_edge_list = []
        for i in range(total_graphs):
            batch_edge_list.append(edge_index + (i * num_nodes))
        batched_edge_index = torch.cat(batch_edge_list, dim=1)

        # Apply Graph Convolution
        spatial_out = self.spatial_gcn(x_gcn_in, batched_edge_index) # [Total_Nodes, Output_Channels]
        spatial_out = spatial_out.view(batch_size, time_steps, num_nodes, -1)

        # 2. Reshape tensor for Temporal One-Dimensional Convolution over time step sequence
        # Target shape for Conv1d: [Batch_Size * Number_of_Nodes, Output_Channels, Time_Steps]
        temporal_in = spatial_out.permute(0, 2, 3, 1).contiguous()
        temporal_in = temporal_in.view(batch_size * num_nodes, -1, time_steps)
        
        temporal_out = self.temporal_conv(temporal_in)
        
        # Reshape back to standard structural ordering: [Batch_Size, Time_Steps, Number_of_Nodes, Output_Channels]
        output = temporal_out.view(batch_size, num_nodes, -1, time_steps)
        output = output.permute(0, 3, 1, 2).contiguous()

        return output


class LatentMotionDiffusionPredictor(nn.Module):
    """
    Conditional Latent Diffusion network forecasting future pose sequences.
    """
    def __init__(self, node_dim: int = 3, hidden_dim: int = 64, forecast_horizon: int = 10):
        super(LatentMotionDiffusionPredictor, self).__init__()
        self.forecast_horizon = forecast_horizon
        self.hidden_dim = hidden_dim

        # Encoder mapping joint trajectories to latent context vector via Spatial-Temporal Graph Neural Network
        self.st_gnn = SpatialTemporalGraphConv(in_channels=node_dim, out_channels=hidden_dim)
        
        # Context projector
        self.context_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True)
        )

        # Noise Prediction Architecture (Epsilon Theta Network)
        self.denoise_net = nn.Sequential(
            nn.Linear(node_dim + hidden_dim + 1, 128), # Input: Noisy Trajectory + ST-GNN Context + Time Step
            nn.ReLU(inplace=True),
            nn.Linear(128, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, node_dim)
        )

    def forward(self, history_poses: torch.Tensor, edge_index: torch.Tensor, diffusion_step: torch.Tensor) -> torch.Tensor:
        """
        Args:
            history_poses: Past pose sequence [Batch_Size, Historical_Steps, Number_of_Nodes, 3]
            edge_index: Topology connectivity [2, Number_of_Edges]
            diffusion_step: Noise variance timestep [Batch_Size, 1]
        Returns:
            Forecasted Trajectories: Probabilistic future poses [Batch_Size, Forecast_Horizon, Number_of_Nodes, 3]
        """
        batch_size, hist_steps, num_nodes, node_dim = history_poses.shape

        # Encode historical motion context via Spatial-Temporal Graph Neural Network
        st_features = self.st_gnn(history_poses, edge_index)           # [Batch_Size, Historical_Steps, Number_of_Nodes, Hidden_Dimension]
        context_vector = st_features.mean(dim=1)                        # Temporal pooling -> [Batch_Size, Number_of_Nodes, Hidden_Dimension]
        context_vector = self.context_proj(context_vector)

        # Sample initial Gaussian noise trajectory x_T ~ Normal Distribution(Mean=0, Variance=Identity)
        noisy_trajectory = torch.randn(
            batch_size, self.forecast_horizon, num_nodes, node_dim, 
            device=history_poses.device
        )

        # Expand context and diffusion timesteps across the forecast horizon
        # Shape: [Batch_Size, Forecast_Horizon, Number_of_Nodes, Hidden_Dimension]
        context_expanded = context_vector.unsqueeze(1).repeat(1, self.forecast_horizon, 1, 1)
        
        # Expand noise timestep: [Batch_Size, Forecast_Horizon, Number_of_Nodes, 1]
        step_expanded = diffusion_step.view(batch_size, 1, 1, 1).repeat(1, self.forecast_horizon, num_nodes, 1)

        # Concatenate features along feature dimension for noise prediction
        net_input = torch.cat([noisy_trajectory, context_expanded, step_expanded], dim=-1)
        
        # Predict spatial noise vector
        predicted_noise = self.denoise_net(net_input)

        # Single-step reverse diffusion update equation: x_{t-1} = x_t - gamma * epsilon_theta
        gamma = 0.05
        denoised_trajectory = noisy_trajectory - (gamma * predicted_noise)

        return denoised_trajectory


# Verification Block
if __name__ == "__main__":
    print("[Module 2 Verification]")
    
    # Define joint kinematic connectivity graph (10 nodes, 9 edges)
    edge_index = torch.tensor([
        [0, 1, 2, 3, 4, 5, 6, 7, 8],
        [1, 2, 3, 4, 5, 6, 7, 8, 9]
    ], dtype=torch.long)

    # Input: Batch of 2, 5 historical frames, 10 nodes, Three-Dimensional coordinates
    history_poses = torch.randn(2, 5, 10, 3)
    timesteps = torch.tensor([[0.2], [0.5]])

    predictor = LatentMotionDiffusionPredictor(node_dim=3, hidden_dim=64, forecast_horizon=10)
    future_prediction = predictor(history_poses, edge_index, timesteps)

    print(f"  Historical Motion Input: {history_poses.shape}")
    print(f"  Forecasted Poses Output: {future_prediction.shape}\n")
    

C:\Users\HP\Documents\hri_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Module 2 Verification]
  Historical Motion Input: torch.Size([2, 5, 10, 3])
  Forecasted Poses Output: torch.Size([2, 10, 10, 3])



In [4]:
"""
Module 3: Model-Based Actor-Critic Policy with Control Barrier Functions
-------------------------------------------------------------------------
Provides:
1. Actor-Critic Network for joint velocity control and state-value estimation.
2. Control Barrier Function enforcing safety constraints (Force_contact < 15 Newtons).
3. Active Safety Filtering Shield protecting human space during dynamic interactions.
"""

import torch
import torch.nn as nn
import torch.optim as optim


class ControlBarrierFunctionLayer(nn.Module):
    """
    Mathematical Control Barrier Function maintaining forward invariance over safe set C:
    
        C = { x in Real Space R^n : h(x) >= 0 }
        
    Defines the physical barrier function index h(x):
        h(x) = minimum( Maximum_Force_Limit - Predicted_Force, Current_Distance - Minimum_Distance_Limit )
    
    If h(x) >= 0, system states remain safe. If h(x) < 0, a safety violation occurs.
    """
    def __init__(self, max_force_limit: float = 15.0, min_distance_limit: float = 0.05):
        super(ControlBarrierFunctionLayer, self).__init__()
        self.max_force_limit = max_force_limit
        self.min_distance_limit = min_distance_limit

    def forward(self, estimated_force: torch.Tensor, estimated_distance: torch.Tensor) -> torch.Tensor:
        """
        Args:
            estimated_force: Scalar tensor representing anticipated contact force (Newtons)
            estimated_distance: Scalar tensor representing Euclidean distance to human surface (meters)
        Returns:
            h_x: Safety barrier index tensor [Batch_Size, 1]
        """
        # Barrier component 1: Force limit constraint
        h_force = self.max_force_limit - estimated_force

        # Barrier component 2: Proximity boundary constraint
        h_distance = estimated_distance - self.min_distance_limit

        # Evaluate minimum boundary constraint index
        h_x = torch.min(h_force, h_distance)
        return h_x


class ContactAwareActorCritic(nn.Module):
    """
    Reinforcement Learning Policy Network evaluating high-level state representations to return 
    joint actuation vectors alongside state-value bounds.
    """
    def __init__(self, state_dim: int = 64, action_dim: int = 6):
        super(ContactAwareActorCritic, self).__init__()
        
        # Shared State Representation Network
        self.feature_network = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 128),
            nn.LayerNorm(128),
            nn.ReLU(inplace=True)
        )

        # Actor Head: Outputs continuous joint velocity commands bounded [-1.0, 1.0]
        self.actor_head = nn.Sequential(
            nn.Linear(128, action_dim),
            nn.Tanh()
        )

        # Critic Head: Predicts expected cumulative reward (State Value V(s))
        self.critic_head = nn.Linear(128, 1)

    def forward(self, state: torch.Tensor):
        """
        Args:
            state: Concatenated vector [Batch_Size, State_Dimension] containing robot kinematics 
                   and human trajectory latent predictions.
        Returns:
            action: Proposed velocity vector [Batch_Size, Action_Dimension]
            state_value: Critic value estimate [Batch_Size, 1]
        """
        features = self.feature_network(state)
        action = self.actor_head(features)
        state_value = self.critic_head(features)

        return action, state_value


class SafetyShieldedController:
    """
    Execution controller integrating Actor-Critic decision-making with Control Barrier Function safety enforcement.
    """
    def __init__(self, state_dim: int = 64, action_dim: int = 6, learning_rate: float = 3e-4):
        self.policy = ContactAwareActorCritic(state_dim=state_dim, action_dim=action_dim)
        self.cbf_layer = ControlBarrierFunctionLayer(max_force_limit=15.0, min_distance_limit=0.05)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=learning_rate)

    def select_action(
        self, 
        state: torch.Tensor, 
        predicted_force: torch.Tensor, 
        predicted_distance: torch.Tensor
    ) -> torch.Tensor:
        """
        Evaluates policy action and applies safety shielding if Control Barrier Function condition h(x) < 0 is triggered.
        """
        self.policy.eval()
        with torch.no_grad():
            nominal_action, _ = self.policy(state)
            
            # Evaluate current Control Barrier Index
            h_x = self.cbf_layer(predicted_force, predicted_distance)

            # Check for safety boundary infringement
            if torch.any(h_x < 0.0):
                # Apply Quadratic Safety Shielding
                # Scale joint actuation commands to slow approach speed and maintain compliance
                scaling_factor = torch.clamp(1.0 + h_x, min=0.01, max=1.0)
                safe_action = nominal_action * scaling_factor
                
                print(f"  [SAFETY SHIELD ACTIVE] Control Barrier Function Violation Index h(x) = {h_x.item():.4f}")
                print(f"  Modulating action magnitude by factor: {scaling_factor.item():.4f}")
                return safe_action

        return nominal_action


# Verification Block
if __name__ == "__main__":
    print("[Module 3 Verification]")
    
    controller = SafetyShieldedController(state_dim=64, action_dim=6)
    
    # Latent state combining robot joint values + predicted human positions
    dummy_state = torch.randn(1, 64)

    # Test Case A: Safe interaction scenario
    print("Scenario A: Safe Distance (0.25 meters), Low Force (3.0 Newtons)")
    action_a = controller.select_action(
        dummy_state, 
        predicted_force=torch.tensor([[3.0]]), 
        predicted_distance=torch.tensor([[0.25]])
    )
    print(f"  Executed Command: {action_a.numpy().round(3)}\n")

    # Test Case B: Boundary violation scenario
    print("Scenario B: Close Distance (0.01 meters), High Force (18.5 Newtons)")
    action_b = controller.select_action(
        dummy_state, 
        predicted_force=torch.tensor([[18.5]]), 
        predicted_distance=torch.tensor([[0.01]])
    )
    print(f"  Executed Command: {action_b.numpy().round(3)}\n")

[Module 3 Verification]
Scenario A: Safe Distance (0.25 meters), Low Force (3.0 Newtons)
  Executed Command: [[0.22  0.591 0.429 0.258 0.226 0.13 ]]

Scenario B: Close Distance (0.01 meters), High Force (18.5 Newtons)
  [SAFETY SHIELD ACTIVE] Control Barrier Function Violation Index h(x) = -3.5000
  Modulating action magnitude by factor: 0.0100
  Executed Command: [[0.002 0.006 0.004 0.003 0.002 0.001]]



In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv

# ==========================================
# 1. MeshPose Estimator Module
# (Monocular 3D Human Mesh & Continuous UV Surface Coordinate Recovery)
# ==========================================

class DifferentiableSpatialSoftmax(nn.Module):
    """
    Computes sub-pixel Two-Dimensional (2D) spatial coordinates from heatmap spatial distributions.
    
    Standard discrete argmax operations are non-differentiable and introduce quantization errors.
    Spatial Softmax calculates the expected continuous (x, y) coordinates via spatial probability maps:
        E[x] = sum_i sum_j (x_j * P(y_i, x_j))
        E[y] = sum_i sum_j (y_i * P(y_i, x_j))
    where P is the spatial Softmax probability distribution over the input heatmaps.
    """
    def __init__(self, height: int, width: int):
        super(DifferentiableSpatialSoftmax, self).__init__()
        self.height = height
        self.width = width

        # Construct coordinate grids normalized to the continuous domain [0.0, 1.0]
        grid_x = torch.linspace(0.0, 1.0, width)
        grid_y = torch.linspace(0.0, 1.0, height)
        
        # Shape: [Height, Width]
        grid_y, grid_x = torch.meshgrid(grid_y, grid_x, indexing='ij')
        
        # Register fixed coordinate grids as non-trainable buffers with shape [1, 1, Height, Width]
        self.register_buffer('grid_x', grid_x.unsqueeze(0).unsqueeze(0))
        self.register_buffer('grid_y', grid_y.unsqueeze(0).unsqueeze(0))

    def forward(self, heatmaps: torch.Tensor) -> torch.Tensor:
        """
        Args:
            heatmaps: Tensor of shape [Batch_Size, Number_of_Vertices, Height, Width]
        Returns:
            Sub-pixel 2D coordinates tensor of shape [Batch_Size, Number_of_Vertices, 2] containing (x, y)
        """
        batch_size, num_vertices, h, w = heatmaps.shape

        # Flatten spatial dimensions to perform continuous Softmax normalization
        flat_heatmaps = heatmaps.view(batch_size, num_vertices, -1)
        probs = F.softmax(flat_heatmaps, dim=-1).view(batch_size, num_vertices, h, w)

        # Compute spatial expectation over normalized coordinate grids
        expected_x = torch.sum(probs * self.grid_x, dim=[-2, -1])  # Shape: [Batch_Size, Number_of_Vertices]
        expected_y = torch.sum(probs * self.grid_y, dim=[-2, -1])  # Shape: [Batch_Size, Number_of_Vertices]

        # Stack expected coordinates into an explicit 2D tensor [Batch_Size, Number_of_Vertices, 2]
        return torch.stack([expected_x, expected_y], dim=-1)


class DenseMeshExtractor(nn.Module):
    """
    Backbone Convolutional Neural Network (CNN) pipeline predicting 2D sub-pixel keypoints
    and continuous UV (Horizontal-Vertical) surface coordinate maps (DensePose paradigm).
    """
    def __init__(self, in_channels: int = 3, num_vertices: int = 689, num_parts: int = 24):
        super(DenseMeshExtractor, self).__init__()
        
        # Feature Extraction Backbone: Downsamples 224x224 input images to 56x56 spatial feature maps
        self.backbone = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False),  # -> [64, 112, 112]
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),                              # -> [64, 56, 56]

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=False),           # -> [128, 56, 56]
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1, bias=False),          # -> [256, 56, 56]
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        # Heatmap Projection Head: Maps internal feature representation to per-vertex confidence heatmaps
        self.heatmap_head = nn.Conv2d(256, num_vertices, kernel_size=1)
        
        # Sub-pixel coordinate extractor operating at spatial resolution (56 x 56)
        self.soft_argmax = DifferentiableSpatialSoftmax(height=56, width=56)
        
        # Dense Surface Projection Head: Predicts 24 anatomical part masks + 2 continuous surface UV channels
        self.uv_head = nn.Conv2d(256, num_parts + 2, kernel_size=1)

    def forward(self, x: torch.Tensor):
        """
        Args:
            x: Raw input Red-Green-Blue (RGB) image tensor [Batch_Size, 3, 224, 224]
        Returns:
            coords_2d: Continuous 2D vertex spatial coordinates [Batch_Size, Number_of_Vertices, 2]
            uv_surface_map: Body part segmentations and UV coordinates [Batch_Size, 26, 56, 56]
        """
        features = self.backbone(x)                  # Intermediate tensor: [Batch_Size, 256, 56, 56]
        heatmaps = self.heatmap_head(features)        # Spatial heatmaps:    [Batch_Size, Number_of_Vertices, 56, 56]
        coords_2d = self.soft_argmax(heatmaps)        # Continuous 2D poses: [Batch_Size, Number_of_Vertices, 2]
        uv_surface_map = self.uv_head(features)       # Continuous UV map:   [Batch_Size, 26, 56, 56]
        return coords_2d, uv_surface_map


class MonocularDepthLifter(nn.Module):
    """
    Lifts estimated 2D image-plane coordinates (x, y) into Three-Dimensional (3D) camera coordinates (x, y, z).
    """
    def __init__(self, num_vertices: int = 689):
        super(MonocularDepthLifter, self).__init__()
        self.num_vertices = num_vertices
        
        # Multi-Layer Perceptron (MLP) regressing non-linear relative depth offsets (z-coordinates)
        self.lifter = nn.Sequential(
            nn.Linear(num_vertices * 2, 1024),
            nn.LayerNorm(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.ReLU(inplace=True),
            
            nn.Linear(512, num_vertices) # Output: Per-vertex depth values z
        )

    def forward(self, coords_2d: torch.Tensor) -> torch.Tensor:
        """
        Args:
            coords_2d: Flattened 2D coordinates [Batch_Size, Number_of_Vertices, 2]
        Returns:
            3D mesh coordinates tensor [Batch_Size, Number_of_Vertices, 3] containing (x, y, z)
        """
        batch_size = coords_2d.size(0)
        flat_2d = coords_2d.view(batch_size, -1)               # Flatten input: [Batch_Size, Number_of_Vertices * 2]
        pred_z = self.lifter(flat_2d).unsqueeze(-1)            # Estimate depth z: [Batch_Size, Number_of_Vertices, 1]
        return torch.cat([coords_2d, pred_z], dim=-1)          # Concatenate along coordinate dim -> [Batch_Size, Number_of_Vertices, 3]


# ==========================================
# 2. Spatial-Temporal Graph & Diffusion Module
# (Spatial-Temporal Graph Neural Networks & Latent Motion Diffusion Forecasting)
# ==========================================

class SpatialTemporalGraphConv(nn.Module):
    """
    Combines Spatial Graph Convolutional Networks (GCN) over spatial skeletal topologies with 1D Temporal Convolutions.
    """
    def __init__(self, in_channels: int, out_channels: int, temporal_kernel_size: int = 3):
        super(SpatialTemporalGraphConv, self).__init__()
        
        # Spatial Graph Convolution Layer operating over anatomical connectivity graphs
        self.spatial_gcn = GCNConv(in_channels, out_channels)
        
        # Temporal Convolutional Layer processing trajectory sequences across time steps
        padding = (temporal_kernel_size - 1) // 2
        self.temporal_conv = nn.Sequential(
            nn.Conv1d(out_channels, out_channels, kernel_size=temporal_kernel_size, padding=padding),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input spatio-temporal tensor [Batch_Size, Time_Steps, Number_of_Nodes, Feature_Dimension]
            edge_index: Graph connectivity matrix [2, Number_of_Edges]
        Returns:
            Output feature representation [Batch_Size, Time_Steps, Number_of_Nodes, Output_Channels]
        """
        batch_size, time_steps, num_nodes, feature_dim = x.shape
        
        # Reshape input tensor to process spatial graphs independently per time step
        x_flat = x.view(batch_size * time_steps, num_nodes, feature_dim)
        x_gcn_in = x_flat.view(-1, feature_dim) # Shape: [Batch_Size * Time_Steps * Number_of_Nodes, Feature_Dimension]
        
        # Construct dynamic batched edge indices across all spatial graphs in the sequence
        total_graphs = batch_size * time_steps
        batch_edge_list = [edge_index + (i * num_nodes) for i in range(total_graphs)]
        batched_edge_index = torch.cat(batch_edge_list, dim=1)

        # Apply Spatial Graph Convolution
        spatial_out = self.spatial_gcn(x_gcn_in, batched_edge_index)
        spatial_out = spatial_out.view(batch_size, time_steps, num_nodes, -1)

        # Permute dimensions for 1D Temporal Convolution: [Batch_Size * Number_of_Nodes, Output_Channels, Time_Steps]
        temporal_in = spatial_out.permute(0, 2, 3, 1).contiguous().view(batch_size * num_nodes, -1, time_steps)
        temporal_out = self.temporal_conv(temporal_in)
        
        # Permute back to spatial-temporal shape: [Batch_Size, Time_Steps, Number_of_Nodes, Output_Channels]
        output = temporal_out.view(batch_size, num_nodes, -1, time_steps).permute(0, 3, 1, 2).contiguous()
        return output


class LatentMotionDiffusionPredictor(nn.Module):
    """
    Conditional Latent Diffusion network forecasting future human trajectory sequences 
    given historical motion context extracted by a Spatial-Temporal Graph Neural Network (ST-GNN).
    """
    def __init__(self, node_dim: int = 3, hidden_dim: int = 64, forecast_horizon: int = 10):
        super(LatentMotionDiffusionPredictor, self).__init__()
        self.forecast_horizon = forecast_horizon
        
        # Spatio-Temporal feature extractor
        self.st_gnn = SpatialTemporalGraphConv(in_channels=node_dim, out_channels=hidden_dim)
        
        # Context projection head mapping temporal features to latent representations
        self.context_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), 
            nn.ReLU(inplace=True)
        )
        
        # Denoising Network (Epsilon Theta): Predicts added spatial noise given noisy inputs, context, and diffusion timestep
        self.denoise_net = nn.Sequential(
            nn.Linear(node_dim + hidden_dim + 1, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, node_dim)
        )

    def forward(self, history_poses: torch.Tensor, edge_index: torch.Tensor, diffusion_step: torch.Tensor) -> torch.Tensor:
        """
        Args:
            history_poses: Historical pose sequence [Batch_Size, History_Steps, Number_of_Nodes, 3]
            edge_index: Spatial connectivity graph indices [2, Number_of_Edges]
            diffusion_step: Scalar diffusion noise step tensor [Batch_Size, 1]
        Returns:
            Forecasted motion trajectories tensor [Batch_Size, Forecast_Horizon, Number_of_Nodes, 3]
        """
        batch_size, hist_steps, num_nodes, node_dim = history_poses.shape
        
        # Extract spatio-temporal representations from historical trajectory sequence
        st_features = self.st_gnn(history_poses, edge_index)
        
        # Aggregate temporal sequence via mean pooling to derive motion context vector: [Batch_Size, Number_of_Nodes, Hidden_Dim]
        context_vector = self.context_proj(st_features.mean(dim=1))
        
        # Sample standard Gaussian noise trajectory x_T ~ Normal Distribution(0, I)
        noisy_trajectory = torch.randn(batch_size, self.forecast_horizon, num_nodes, node_dim, device=history_poses.device)
        
        # Replicate temporal context across the forecasting horizon: [Batch_Size, Forecast_Horizon, Number_of_Nodes, Hidden_Dim]
        context_expanded = context_vector.unsqueeze(1).repeat(1, self.forecast_horizon, 1, 1)
        
        # Replicate scalar noise timesteps: [Batch_Size, Forecast_Horizon, Number_of_Nodes, 1]
        step_expanded = diffusion_step.view(batch_size, 1, 1, 1).repeat(1, self.forecast_horizon, num_nodes, 1)

        # Concatenate noisy trajectory states, conditioning context, and diffusion steps along feature dimension
        net_input = torch.cat([noisy_trajectory, context_expanded, step_expanded], dim=-1)
        
        # Predict spatial noise vector
        predicted_noise = self.denoise_net(net_input)
        
        # Apply single-step reverse diffusion update: x_{t-1} = x_t - gamma * epsilon_theta
        return noisy_trajectory - (0.05 * predicted_noise)


# ==========================================
# 3. Contact Safe Reinforcement Learning Module
# (Reinforcement Learning Policy with Active Control Barrier Function Shielding)
# ==========================================

class ControlBarrierFunctionLayer(nn.Module):
    """
    Evaluates safety barrier index h(x) to maintain forward invariance over safe operational sets:
        Set C = { x in Real Space R^n : h(x) >= 0 }
    
    If h(x) >= 0, physical constraints are satisfied. If h(x) < 0, a safety violation occurs.
    """
    def __init__(self, max_force_limit: float = 15.0, min_distance_limit: float = 0.05):
        super(ControlBarrierFunctionLayer, self).__init__()
        self.max_force_limit = max_force_limit          # Upper limit on contact interaction force (Newtons)
        self.min_distance_limit = min_distance_limit    # Minimum allowable proximity distance (meters)

    def forward(self, estimated_force: torch.Tensor, estimated_distance: torch.Tensor) -> torch.Tensor:
        """
        Args:
            estimated_force: Anticipated interaction contact force tensor (Newtons)
            estimated_distance: Distance relative to human body surface (meters)
        Returns:
            h_x: Control Barrier Index tensor evaluating structural safety margin
        """
        # Barrier metric 1: Force containment constraint
        h_force = self.max_force_limit - estimated_force
        
        # Barrier metric 2: Proximity constraint
        h_distance = estimated_distance - self.min_distance_limit
        
        # Compute critical safety bound as the minimum constraint value
        return torch.min(h_force, h_distance)


class ContactAwareActorCritic(nn.Module):
    """
    Actor-Critic architecture predicting joint actuation vectors (Actor) 
    and estimating expected state values V(s) (Critic).
    """
    def __init__(self, state_dim: int = 64, action_dim: int = 6):
        super(ContactAwareActorCritic, self).__init__()
        
        # Shared feature extraction backbone
        self.feature_network = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 128),
            nn.LayerNorm(128),
            nn.ReLU(inplace=True)
        )
        
        # Actor Head: Outputs continuous joint velocity control bounded in [-1.0, 1.0] via Tanh activation
        self.actor_head = nn.Sequential(
            nn.Linear(128, action_dim), 
            nn.Tanh()
        )
        
        # Critic Head: Estimates continuous state-value baseline V(s)
        self.critic_head = nn.Linear(128, 1)

    def forward(self, state: torch.Tensor):
        """
        Args:
            state: Concatenated system state vector [Batch_Size, State_Dimension]
        Returns:
            nominal_action: Proposed unshielded joint velocities [Batch_Size, Action_Dimension]
            state_value: Expected cumulative baseline reward [Batch_Size, 1]
        """
        features = self.feature_network(state)
        return self.actor_head(features), self.critic_head(features)


class SafetyShieldedController:
    """
    Execution controller integrating model-based Actor-Critic inference with 
    Control Barrier Function (CBF) active safety shielding.
    """
    def __init__(self, state_dim: int = 64, action_dim: int = 6, learning_rate: float = 3e-4):
        self.policy = ContactAwareActorCritic(state_dim=state_dim, action_dim=action_dim)
        self.cbf_layer = ControlBarrierFunctionLayer(max_force_limit=15.0, min_distance_limit=0.05)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=learning_rate)

    def select_action(self, state: torch.Tensor, predicted_force: torch.Tensor, predicted_distance: torch.Tensor) -> torch.Tensor:
        """
        Evaluates nominal policy actions and applies continuous quadratic safety modulation 
        if the Control Barrier Function triggers a boundary violation (h(x) < 0).
        """
        self.policy.eval()
        with torch.no_grad():
            nominal_action, _ = self.policy(state)
            
            # Evaluate Control Barrier Index h(x)
            h_x = self.cbf_layer(predicted_force, predicted_distance)
            
            # Active Shielding Intervention: Intercept dangerous velocity commands when h(x) < 0
            if torch.any(h_x < 0.0):
                # Scale control actions dynamically to enforce compliance and lower approach velocity
                scaling_factor = torch.clamp(1.0 + h_x, min=0.01, max=1.0)
                safe_action = nominal_action * scaling_factor
                
                print(f"  [SAFETY SHIELD ACTIVE] Control Barrier Function Index h(x) = {h_x.item():.4f}")
                print(f"  Modulating action magnitude by factor: {scaling_factor.item():.4f}")
                return safe_action
                
        return nominal_action


# ==========================================
# 4. Pipeline Execution
# (End-to-End System Integration Script)
# ==========================================

def run_pipeline():
    print("==========================================================")
    print(" Starting Full Human-Centric Physical Artificial Intelligence Pipeline")
    print("==========================================================")

    # Dimensionality parameters
    num_vertices, hist_steps, num_nodes = 689, 5, 10
    
    # Instantiate pipeline components
    mesh_extractor = DenseMeshExtractor(num_vertices=num_vertices)
    depth_lifter = MonocularDepthLifter(num_vertices=num_vertices)
    diffusion_predictor = LatentMotionDiffusionPredictor(node_dim=3, hidden_dim=64, forecast_horizon=10)
    rl_controller = SafetyShieldedController(state_dim=64, action_dim=6)

    # 1. Simulate incoming monocular Red-Green-Blue (RGB) image stream [Batch=1, Channels=3, H=224, W=224]
    raw_rgb_frame = torch.randn(1, 3, 224, 224)
    print("\n[Step 1: Visual Perception]")
    coords_2d, uv_maps = mesh_extractor(raw_rgb_frame)
    full_3d_mesh = depth_lifter(coords_2d)
    print(f"  Estimated Three-Dimensional Human Mesh Output: {full_3d_mesh.shape}")

    # 2. Extract primary tracking joints and simulate a temporal history window (5 past frames)
    selected_joints = full_3d_mesh[:, :num_nodes, :]
    historical_poses = selected_joints.unsqueeze(1).repeat(1, hist_steps, 1, 1)
    
    # Define joint connectivity topology graph (10 joints, 9 edges)
    edge_index = torch.tensor([[0, 1, 2, 3, 4, 5, 6, 7, 8], [1, 2, 3, 4, 5, 6, 7, 8, 9]], dtype=torch.long)
    diffusion_step = torch.tensor([[0.1]])

    # 3. Forecast future trajectory sequence over 10 future steps using Motion Diffusion
    print("\n[Step 2: Motion Diffusion Forecasting]")
    forecasted_trajectories = diffusion_predictor(historical_poses, edge_index, diffusion_step)
    print(f"  Forecasted Future Pose Sequence Output: {forecasted_trajectories.shape}")

    # 4. Safe Reinforcement Learning (RL) Execution & Safety Shielding
    print("\n[Step 3: Safe Policy Execution & Control Barrier Function Shielding]")
    
    # Aggregate spatio-temporal predictions into unified latent state representation [1, 64]
    latent_state = torch.cat([forecasted_trajectories.mean(dim=[1, 2]), torch.zeros(1, 61)], dim=-1)

    # Test safety intervention: Force of 16.2N violates the safety boundary limit of 15.0N
    simulated_force = torch.tensor([[16.2]])
    simulated_distance = torch.tensor([[0.03]])

    # Evaluate policy action with Control Barrier Function shielding
    joint_velocity_command = rl_controller.select_action(
        state=latent_state,
        predicted_force=simulated_force,
        predicted_distance=simulated_distance
    )

    print(f"\nFinal Executed Robot Joint Action Vector:\n  {joint_velocity_command.numpy().round(4)}")
    print("==========================================================")
    print(" Pipeline Execution Completed Successfully.")
    print("==========================================================")

# Entry point for single-cell notebook environments
run_pipeline()

 Starting Full Human-Centric Physical Artificial Intelligence Pipeline

[Step 1: Visual Perception]
  Estimated Three-Dimensional Human Mesh Output: torch.Size([1, 689, 3])

[Step 2: Motion Diffusion Forecasting]
  Forecasted Future Pose Sequence Output: torch.Size([1, 10, 10, 3])

[Step 3: Safe Policy Execution & Control Barrier Function Shielding]
  [SAFETY SHIELD ACTIVE] Control Barrier Function Index h(x) = -1.2000
  Modulating action magnitude by factor: 0.0100

Final Executed Robot Joint Action Vector:
  [[ 0.0042  0.0008  0.0018  0.0064 -0.0068  0.0027]]
 Pipeline Execution Completed Successfully.
